[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kewel-corp/book-vol1/blob/main/notebooks/ch34_code.ipynb)

# 第34章　学習ループを実装する：最良モデルの保存・再開・記録

**『医療診断支援AI開発　第1巻　入門編 ― 動かして、自分で作る（第1巻　入門編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol1

## 34.1　学習ループの全体

In [ ]:
import torch, copy, math
from torch.amp import autocast, GradScaler

SELECT_KEY = "ap"    # 25.3と同じ。評価辞書のどのキーで最良を選ぶか（本章のコードでも定義しておく）

def train(model, train_loader, val_loader, evaluate, device,
          epochs=100, lr=1e-4, patience=15, ckpt="best.pth"):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = torch.nn.CrossEntropyLoss()
    dev = torch.device(device).type                 # "cuda" か "cpu" か
    amp_on = (dev == "cuda")                        # CPUでは混合精度を無効にする（不要な警告を防ぐ）
    scaler = GradScaler(dev, enabled=amp_on)
    # 最良スコアは -inf から始める。0.0 で始めると、スコアが0のときや負の指標を
    # 使うとき、best_state が初期重みのまま「学習済み」として返る。
    # best_state は初期重みで埋めておき、load_state_dict(None) でのクラッシュも防ぐ。
    best_score, best_state, no_improve = float("-inf"), copy.deepcopy(model.state_dict()), 0
    got_valid_score = False        # 一度も有効な評価が得られなければ、明示的に失敗させる

    for epoch in range(epochs):
        # --- 学習フェーズ ---
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            with autocast(dev, enabled=amp_on):        # 混合精度（CPUでは自動的に無効）
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
        cur_lr = optimizer.param_groups[0]["lr"]        # このエポックで実際に使った学習率
        scheduler.step()                                # 学習率を更新（次エポックぶん）

        # --- 検証フェーズ ---
        m = evaluate(model, val_loader, device)         # 25.3節の evaluate と同じ入出力仕様（指標の辞書）
        score = m[SELECT_KEY]                           # APなど事前に定めた指標で選ぶ（感度単独では選ばない）
        if not math.isfinite(score):                    # 評価不能をスコア0として扱わない
            print(f"epoch {epoch}: 評価不能（{SELECT_KEY} が有限値でない）")
            continue
        got_valid_score = True
        print(f"epoch {epoch}: val {SELECT_KEY}={score:.4f} lr={cur_lr:.2e}")

        # --- 最良の保存と早期終了 ---
        if score > best_score:
            best_score = float(score)            # NumPy の数値のまま保存すると、新しい PyTorch の torch.load（weights_only）で読めない
            best_state = copy.deepcopy(model.state_dict())
            torch.save({"epoch": epoch, "model": best_state,
                        "score": best_score}, ckpt)     # チェックポイント
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:                  # 早期終了
                print(f"早期終了（{patience}エポック改善なし）")
                break

    if not got_valid_score:                             # 有効な検証なしに初期重みを返さない
        raise RuntimeError("有効な検証スコアが一度も得られなかった")
    model.load_state_dict(best_state)                   # 最良を復元して返す
    return model, best_score

## 34.3　中断からの再開

In [ ]:
ckpt = torch.load("best.pth", map_location=device)
model.load_state_dict(ckpt["model"])
start_epoch = ckpt["epoch"] + 1              # 続きから
print(f"epoch {start_epoch} から再開（前回スコア {ckpt['score']:.4f}）")

## 切断に備えてチェックポイントをGoogle Driveに残す

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")                       # 初回は認証を求められる
CKPT_DIR = "/content/drive/MyDrive/medai_ckpt"      # ここは再起動後も残る
os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
import torch
def save_ckpt(state, path):
    tmp = path + ".tmp"
    torch.save(state, tmp)          # まず一時ファイルへの書き込みを完了する
    os.replace(tmp, path)           # 書き終えてから一度に置き換える（書き込み途中のファイルで上書きしない）

In [ ]:
# 最良判定と保存は、この1か所にまとめる。順序が大事で、先に最良を判定して履歴を更新し、
# その上で last を保存する。逆にすると、last には「更新前の best_score」が入り、再開時に巻き戻る。
got_valid_score = True                      # score が有限値かどうかは、このブロックの前で確かめておく
improved = score > best_score
if improved:
    best_score = float(score)
    best_state = copy.deepcopy(model.state_dict())   # 最良の重みは「いまの重み」と別に持つ
    no_improve = 0
else:
    no_improve += 1
state = {
    "epoch": epoch,
    "model": model.state_dict(),           # 再開に使う「いまの重み」
    "optimizer": optimizer.state_dict(),   # 再開後の学習率・モメンタムの状態を引き継ぐために保存する
    "scheduler": scheduler.state_dict(),
    "scaler": scaler.state_dict(),         # AMPのスケーラも残す
    "best_state": best_state,              # 最後に返す「最良の重み」
    "best_score": best_score,
    "got_valid_score": got_valid_score,
    "no_improve": no_improve,              # 早期終了のカウンタ
    "classes": train_ds.classes,           # 予測番号→ラベル名（後で結果を解釈するために記録する）
    "img_size": 224,
    "normalize": {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
}
if improved:
    save_ckpt(state, f"{CKPT_DIR}/best.pth")   # 改善時だけ（推論用）
save_ckpt(state, f"{CKPT_DIR}/last.pth")       # 毎エポック（再開用）
# 早期終了（no_improve >= patience で break）の判定は、保存の後で行う。先に抜けると今回の履歴が残らない

In [ ]:
# 続きがない初回の既定値（未定義のまま学習ループに入らない）
start_epoch, best_score, best_state = 0, float("-inf"), None
got_valid_score, no_improve = False, 0
path = f"{CKPT_DIR}/last.pth"
if os.path.exists(path):
    ck = torch.load(path, map_location=device, weights_only=False)
    required = {"best_state", "best_score", "got_valid_score", "no_improve", "scaler"}
    if not required.issubset(ck):
        # 最良の履歴を持たない古い形式。best.pth が同じ実験の最良と対になっているか確かめてから移す
        raise ValueError(f"{path} に最良の履歴がない（古い形式）。best.pth と照合してから移行すること")
    model.load_state_dict(ck["model"])          # 学習を続けるのは「いまの重み」から
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    scaler.load_state_dict(ck["scaler"])        # AMPのスケーラ
    start_epoch = ck["epoch"] + 1
    best_state = ck["best_state"]               # 返すのは「最良の重み」。いまの重みで埋めない
    best_score = float(ck["best_score"])        # 表示するだけでなく、変数へ戻す
    got_valid_score = bool(ck["got_valid_score"])
    no_improve = int(ck["no_improve"])          # 早期終了のカウンタも引き継ぐ
    if got_valid_score and (best_state is None or not math.isfinite(best_score)):
        raise ValueError("最良の重みとスコアの履歴が食い違っている")
    print(f"Driveの{path} から epoch {start_epoch} で再開（前回best {best_score:.4f}）")
for epoch in range(start_epoch, epochs):
    ...                                          # 本章の train ループ本体（最良判定と保存は上の1か所）
if not got_valid_score or best_state is None:   # 有効な検証なしに初期重みを返さない
    raise RuntimeError("有効な検証結果と最良の重みが得られていない")
model.load_state_dict(best_state)                # 再開後に改善がなくても、過去の最良を返す

## 学習の経過を見ながら進める：TensorBoardとmatplotlibでの監視

In [ ]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter("runs/exp1")          # ログはこのフォルダに記録される
# --- 学習ループ内、各エポック末に ---
# loss.item() は「最終バッチの損失」。エポックの代表値にするなら、学習フェーズで
#   epoch_loss_sum += loss.item() * x.size(0);  n_samples += x.size(0)
# と件数で重み付けて足しておき、ここで割る（クラス重み付きCEなら件数ではなく criterion.weight[y].sum() で重み付ける。25.5の「学習ログを読む」の項を参照）。
writer.add_scalar("loss/train", epoch_loss_sum / n_samples, epoch)
writer.add_scalar(f"{SELECT_KEY}/val", score, epoch)   # 指標名は実際の中身に合わせる
writer.add_scalar("lr", cur_lr, epoch)                 # このエポックで実際に使った学習率
writer.flush()                                # Driveにログを残す場合は runs/ フォルダごと保存する

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs